In [3]:
cgroup <- c(90,95,67,120,89,92,100,82,79,85)
max_target_mean <- 95
paste("Mean blood pressure and standard deviation in the conventional treatment sample:", mean(cgroup), "and", round(sd(cgroup),2))

[1] "Mean blood pressure and standard deviation in the conventional treatment sample: 89.9 and 14.02"

In [4]:
c_mean <- mean(cgroup)
c_sd <- sd(cgroup)

# call the one-sided t-test
t.test(x=cgroup,alternative="less",mu=max_target_mean)


	One Sample t-test

data:  cgroup
t = -1.1504, df = 9, p-value = 0.1398
alternative hypothesis: true mean is less than 95
95 percent confidence interval:
    -Inf 98.0268
sample estimates:
mean of x 
     89.9 


In [5]:
m_sd <- c_sd/sqrt(length(cgroup))
t.value <- (c_mean-max_target_mean)/m_sd
# use the distribution function of t-distribution with 9 degrees of freedom to obtain p-value
paste("P-value of one-sided t-test:", round(pt(t.value,df=9,lower.tail=T),2))

[1] "P-value of one-sided t-test: 0.14"

In [6]:
# the upper bound of the confidence interval
paste("The upper bound of the 95%-confidence interval of the coventional treatment sample mean:", c_mean+qt(0.95,df=9)*m_sd)

[1] "The upper bound of the 95%-confidence interval of the coventional treatment sample mean: 98.0268006681093"

In [7]:
ngroup <- c(71,79,69,98,91,85,89,75,78,80)
paste("Mean blood pressure and standard deviation in the new treatment sample:", mean(ngroup), "and", round(sd(ngroup),2))

[1] "Mean blood pressure and standard deviation in the new treatment sample: 81.5 and 9.19"

In [8]:
t.test(x=ngroup,alternative="less",mu=max_target_mean)



	One Sample t-test

data:  ngroup
t = -4.6441, df = 9, p-value = 0.0006061
alternative hypothesis: true mean is less than 95
95 percent confidence interval:
     -Inf 86.82865
sample estimates:
mean of x 
     81.5 


In [9]:
t.test(x=cgroup,y=ngroup,var.equal=T,alternative="greater")



	Two Sample t-test

data:  cgroup and ngroup
t = 1.5845, df = 18, p-value = 0.06525
alternative hypothesis: true difference in means is greater than 0
95 percent confidence interval:
 -0.7928998        Inf
sample estimates:
mean of x mean of y 
     89.9      81.5 


ANSWERS TO SECOND EXERCISE


1) propose a similar scenario where a two-tailed test could be applied, how would you change the formulas and graph interpretations above?
2) what happens if we decrease/increase the sample size? explain the influence on statistical testing (technical changes, significance and power of the test),
3) mention as many principal changes in the study design as possible that would require choosing a different statistical test.


1) Two tailed tests are used when we are comparing scenarios from two ends. For example in the previous exercise we compared whether the blood pressure was equal to or less than 95, however, in the drug comparison scenario we compared two independent occurences. A scenario similar to the second exercise would be for example if a farmaceutical company was comparing two versions of a drug formulation. They do not know if the first drug is better than the other one, so our hypotheses would be: H0: New drug formulation is the same as the old one. Ha: New drug is different in either direction (better or worse). The key difference would be in benchmark values. Because we are making a two tailed test we split the p-value of 0.05 into two. So 2.5 % in each tail. which means that the |t| has to be larger or equal to 2.1 in order to reject the null hypothesis in favour of the alternative one. 

2) If we increase the sample size for example n=50, the degrees of freedom will be 98 and our critical t value will decrease. This also decreases the standard error. This will also achieve more precise confidence intervals. 

3) The t-test cannot be used if we don't have an iid dataset which means that we violate the independence of samples and identical distribution assumptions. Independence of the samples can be violated by measuring the same patients, identical distribution can be violated when different groups have different variances or distributions. 

In [10]:
mu_c <- 90
mu_n <- 90
sigma_c <- 12
sigma_n <- 12

## sample size and the number of repeats
sample_size <- 10
reps <- 10000

## generate the samples and run the tests
# is mean in cgroup <95?
tcs <- c() 
pcs <- c()
# is mean in ngroup <95?
tns <- c() 
pns <- c()
# compare cgroup and ngroup
tcns <- c() 
pcns <- c()
for (rep in seq(reps)){
  ## generate two samples with the random normal generator
  cgroup <- rnorm(sample_size,mean=mu_c,sd=sigma_c)
  ngroup <- rnorm(sample_size,mean=mu_n,sd=sigma_n)
  ## compare groups with 95
  ## t-test statistic
  tcs[rep] <- (mean(cgroup)-max_target_mean)/(sd(cgroup)/sqrt(sample_size))
  tns[rep] <- (mean(ngroup)-max_target_mean)/(sd(ngroup)/sqrt(sample_size))
  ## do the same with built in t-test (remember only p-values)
  pcs[rep] <- t.test(x=cgroup,alternative="less",mu=max_target_mean)$p.value
  pns[rep] <- t.test(x=ngroup,alternative="less",mu=max_target_mean)$p.value
  ## compare cgroup and ngroup
  ## t-test statistic
  tcns[rep] <- (mean(cgroup)-mean(ngroup))/(sqrt((sd(cgroup)^2+sd(ngroup)^2)/2)*sqrt(2/sample_size))
  ## do the same with built in t-test (remember only p-values), run two-tailed test this time
  pcns[rep] <- t.test(cgroup,ngroup,alternative = "two.sided",var.equal = TRUE,conf.level = 0.95)$p.value
}

POWER OF THE TEST AND TYPE II ERROR


In [17]:
## how often will we correctly reject the null hypothesis?
## the power of test
## can be reached in two ways, to compare the p-values with alpha or the t-statistics with the corresponding t-distribution quantile 
thres_05 <- qt(0.05,df=sample_size-1)
## the conventional group
## the same outcome can be reached with: mean(pcs<0.05)
paste("The experimental power of t-test in the conventional group:", mean(tcs<thres_05))




[1] "The experimental power of t-test in the conventional group: 0.3355"

In [18]:
## the new drug group
## the same outcome can be reached with: mean(pns<0.05)
paste("The experimental power of t-test in the new group:", mean(tns<thres_05))

[1] "The experimental power of t-test in the new group: 0.3356"

In [19]:
## t-statistic that is expected under ideal conditions when the sample mean and variance agree with the population mean and variance
t_expected <- (mu_c-max_target_mean)/(sigma_c/sqrt(sample_size))
## the real power of the test derived from theory
## shift the distribution function to the left by |t_expected|
paste("The theoretical power of t-test in each group:", pt(thres_05-t_expected,df=sample_size-1))


[1] "The theoretical power of t-test in each group: 0.309314170128656"

TYPE I ERROR AND ALPHA

In [14]:
## how often will we incorrectly reject the null hypothesis about the identity of population means?
## Type I error, false positive decisions
thres_025 <- qt(0.975,df=2*sample_size-2)
## the same outcome can be reachjed with: mean(pcns<0.05)
paste("The probability of Type I error of the t-test that compares both the groups:", mean(abs(tcns)>thres_025))

[1] "The probability of Type I error of the t-test that compares both the groups: 0.0492"

1) the power of the test that evaluates whether “the mean blood pressure after treatment is smaller than 95” is relatively low, change the parameters above to increase this power, report when you reach 80%,

In [20]:
# Function to calculate power for given parameters
calculate_power <- function(mu, sigma, target, n, alpha = 0.05) {
  t_expected <- (mu - target) / (sigma / sqrt(n))
  thres <- qt(alpha, df = n - 1)
  power <- pt(thres - t_expected, df = n - 1)
  return(power)
}

# Current power
current_power <- calculate_power(mu_c, sigma_c, max_target_mean, sample_size)
cat("Current power with n=10:", round(current_power, 3), "\n")

Current power with n=10: 0.309 


In [22]:
#Try increase sample size to reach 80% power
sample_sizes <- c(15, 20, 25, 30, 35, 40, 45, 50)
powers_n <- sapply(sample_sizes, function(n) calculate_power(mu_c, sigma_c, max_target_mean, n))

for (i in 1:length(sample_sizes)) {
  cat("Sample size:", sample_sizes[i], "Power:", round(powers_n[i], 3), "\n")
  if (powers_n[i] >= 0.80) {
    cat("Achieved 80% power with sample size:", sample_sizes[i], "\n")
    break
  }
}


Sample size: 15 Power: 0.442 
Sample size: 20 Power: 0.553 
Sample size: 25 Power: 0.644 
Sample size: 30 Power: 0.718 
Sample size: 35 Power: 0.778 
Sample size: 40 Power: 0.826 
Achieved 80% power with sample size: 40 


2) see what happens with the test when violating assumptions with outliers, inject a severe outlier in the sample and explain the outcomes,


In [23]:
set.seed(123)
normal_sample <- rnorm(10, mean = 85, sd = 12)
cat("Normal sample:", round(normal_sample, 2), "\n")
cat("Normal sample mean:", round(mean(normal_sample), 2), "\n")
cat("Normal sample SD:", round(sd(normal_sample), 2), "\n")

# Test without outlier
test_normal <- t.test(normal_sample, alternative = "less", mu = 95)
cat("t-statistic (normal):", round(test_normal$statistic, 3), "\n")
cat("p-value (normal):", round(test_normal$p.value, 4), "\n")

Normal sample: 78.27 82.24 103.7 85.85 86.55 105.58 90.53 69.82 76.76 79.65 
Normal sample mean: 85.9 
Normal sample SD: 11.45 
t-statistic (normal): -2.516 
p-value (normal): 0.0165 


In [24]:
# Add severe outlier
outlier_sample <- c(normal_sample[-1], 150)  # Replace one value with extreme outlier
cat("\nWith outlier (150 instead of", round(normal_sample[1], 2), "):\n")
cat("Sample with outlier:", round(outlier_sample, 2), "\n")
cat("Sample mean:", round(mean(outlier_sample), 2), "\n")
cat("Sample SD:", round(sd(outlier_sample), 2), "\n")

# Test with outlier
test_outlier <- t.test(outlier_sample, alternative = "less", mu = 95)
cat("t-statistic (with outlier):", round(test_outlier$statistic, 3), "\n")
cat("p-value (with outlier):", round(test_outlier$p.value, 4), "\n")

# Compare results
cat("\nComparison:\n")
cat("Mean change:", round(mean(outlier_sample) - mean(normal_sample), 2), "\n")
cat("SD change:", round(sd(outlier_sample) - sd(normal_sample), 2), "\n")
cat("t-statistic change:", round(test_outlier$statistic - test_normal$statistic, 3), "\n")
cat("p-value change:", round(test_outlier$p.value - test_normal$p.value, 4), "\n")


With outlier (150 instead of 78.27 ):
Sample with outlier: 82.24 103.7 85.85 86.55 105.58 90.53 69.82 76.76 79.65 150 
Sample mean: 93.07 
Sample SD: 22.89 
t-statistic (with outlier): -0.267 
p-value (with outlier): 0.3978 

Comparison:
Mean change: 7.17 
SD change: 11.45 
t-statistic change: 2.249 
p-value change: 0.3813 


Outlier effects:
- Increases sample mean and standard deviation
- Reduces test statistic magnitude
- Increases p-value (reduces power to detect true effect)
- Makes test less sensitive to real differences

3) violate other assumptions to see what happens (a distribution different from normal one, unequal variances, etc.),



4) show the setting in which the central limit theorem can be applied, gradually change the setting to see what happens with the distribution of sample means.